# BabyBrain Robust Meta-Training (Paperspace)

Run cells top-to-bottom. This notebook applies a learn2learn source-build workaround, trains the robust Conv4-192 checkpoint on parameterized abstract patterns (no literal icon classes), then evaluates concepts.

In [ ]:
from pathlib import Path

# Change this if your repo is somewhere else on Paperspace.
REPO_PATH = Path("/notebooks/babybrain")
if not (REPO_PATH / "requirements.txt").exists():
    raise FileNotFoundError(f"requirements.txt not found at {REPO_PATH}")

%cd {REPO_PATH}
!pwd
!ls -1 | head

In [ ]:
import subprocess
import sys
from pathlib import Path

repo = Path.cwd()
cmd = f"""
set -euo pipefail
cd {repo}

python -m pip install -U pip setuptools wheel cython
grep -v '^learn2learn' requirements.txt > /tmp/requirements_no_l2l.txt
python -m pip install -r /tmp/requirements_no_l2l.txt

rm -rf /tmp/learn2learn_build
git clone --depth 1 https://github.com/learnables/learn2learn.git /tmp/learn2learn_build
cd /tmp/learn2learn_build
python setup.py build_ext --inplace
python -m pip install .
"""
subprocess.run(["bash", "-lc", cmd], check=True)

import learn2learn
print("learn2learn install OK")

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    "meta_training/train_robust_contracts.py",
    "--hidden", "192",
    "--iterations", "4000",
    "--tasks-per-batch", "8",
    "--support", "8",
    "--query", "8",
    "--inner-steps", "5",
    "--legacy-shape-ratio", "0.35",
    "--instance-ratio", "0.70",
    "--augment-prob", "0.85",
    "--save-name", "general_conv4_192_robust.pt",
]

subprocess.run(cmd, check=True)

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    "meta_training/eval_concepts.py",
    "--checkpoint", "app/models/checkpoints/general_conv4_192_robust.pt",
    "--hidden", "192",
], check=True)

In [ ]:
# Optional: try a larger model only if Conv4-192 plateaus.
# This is slower but still potentially usable for adaptation.
#
# import subprocess, sys
# subprocess.run([
#     sys.executable,
#     "meta_training/train_robust_contracts.py",
#     "--hidden", "256",
#     "--iterations", "4000",
#     "--tasks-per-batch", "8",
#     "--support", "8",
#     "--query", "8",
#     "--inner-steps", "5",
#     "--legacy-shape-ratio", "0.35",
#     "--instance-ratio", "0.70",
#     "--augment-prob", "0.85",
#     "--save-name", "general_conv4_256_robust.pt",
# ], check=True)